# Phase D — Multi-Scale Feature Fusion (`E16_FPN`, `E17_BiFPN`)
**Project:** 22AI304 / 22AIE304 End-Semester Capstone — Team 11  
**Protocol:** Attention-free `PDN-S` + 4-level BN-free multi-scale pyramid (`P2..P5 → 59×59`) under the exact 200-epoch `AdamW` cosine schedule (`patience=10`, native BF16 on A100/L4/G4, FP16 on T4).

- **`E16_FPN`**: Standard unidirectional top-down Feature Pyramid Network (`P5 → P4 → P3 → P2`)
- **`E17_BiFPN`**: Weighted bi-directional Feature Pyramid Network (`P5 → P2` + `P2 → P5` with fast normalized fusion weights & depthwise-separable convs)

**Reference Baselines:** `E01_PUAD_200ep` (`89.64%` Mean / `84.07%` Pooled) | Phase C Winner `E15_CrissCross` (`90.74%` Mean / `85.33%` Pooled).

In [ ]:
#@title 1. Verify Colab GPU, RAM, Disk & Install Dependencies
!pip install -q timm thop pyyaml scikit-learn pandas
import torch, psutil, shutil, sys
print('=== COLAB SESSION AUDIT (PHASE D) ===')
print(f'Python        : {sys.version.split()[0]}')
print(f'PyTorch       : {torch.__version__} | CUDA: {torch.version.cuda}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    native_bf16 = p.major >= 8
    print(f'GPU Model     : {p.name} (CC {p.major}.{p.minor}) | VRAM: {p.total_memory/1024**3:.2f} GiB | Native BF16: {native_bf16}')
else:
    raise RuntimeError('No GPU attached! Runtime -> Change runtime type -> L4 / G4 / A100 GPU.')
ram = psutil.virtual_memory()
disk = shutil.disk_usage('/content')
print(f'System RAM    : {ram.available/1e9:.1f} GB free / {ram.total/1e9:.1f} GB total')
print(f'Local SSD     : {disk.free/1e9:.1f} GB free / {disk.total/1e9:.1f} GB total')

In [ ]:
#@title 2. Mount Drive, Pull Latest Repo, Unpack Data & Run Section 3 Audit
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_URL = 'https://github.com/Chimera418/dl-endsem-logical-ad.git'
DRIVE_ROOT = '/content/drive/MyDrive/DL_Endsem_Team11'
DRIVE_RUNS = f'{DRIVE_ROOT}/runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)

if not os.path.exists('/content/Endsem/.git'):
    !git clone {REPO_URL} /content/Endsem
else:
    !git -C /content/Endsem pull --ff-only
%cd /content/Endsem

if not os.path.exists('/content/data/mvtec_loco_anomaly_detection/breakfast_box'):
    os.makedirs('/content/data/mvtec_loco_anomaly_detection', exist_ok=True)
    !tar -xf "{DRIVE_ROOT}/data/mvtec_loco_anomaly_detection.tar.xz" -C /content/data/mvtec_loco_anomaly_detection/
if not os.path.exists('/content/data/imagenet_penalty'):
    os.makedirs('/content/data/imagenet_penalty', exist_ok=True)
    !unzip -q "{DRIVE_ROOT}/data/imagenet_penalty_10k.zip" -d /content/data/imagenet_penalty

os.environ['ENDSEM_LOCO_ROOT'] = '/content/data/mvtec_loco_anomaly_detection'
os.environ['ENDSEM_IMAGENET_ROOT'] = '/content/data/imagenet_penalty'
os.environ['ENDSEM_TEACHER_PATH'] = f'{DRIVE_ROOT}/weights/teacher_pdn_s.pt'

!python scripts/verify_env.py

In [ ]:
#@title 3. Run Phase D (E16_FPN & E17_BiFPN) — Sequential (Clean Solo Latency & Best on L4)
!python scripts/run_experiment.py --config configs/E16_fpn.yaml --mode train --drive-sync-dir "$DRIVE_RUNS"
!python scripts/run_experiment.py --config configs/E17_bifpn.yaml --mode train --drive-sync-dir "$DRIVE_RUNS"

In [ ]:
#@title 4. Aggregate Phase D vs. Baseline & Phase C Winner Summary Table
!python scripts/aggregate_results.py --dirs experiments "$DRIVE_RUNS" --out-dir "$DRIVE_RUNS/phaseD_summary"